# SAE Clamp Steering (alpha=1, fixed)

Simplified notebook: **clamp method only**, **alpha = 1 for all questions** (hardcoded).

What it does:
- Uncertainty features → push up toward target activation
- Certainty features → suppress toward zero
- `alpha = 1` always, no quantization and no dependence on VU/SE

**Files to upload:**
1. **test.csv** — CSV (`question` + any other columns)
2. **intervention_v2.pt** — config from `build_intervention_config_v2.py`

## 0. Check GPU

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"Python: {sys.version}")

## 1. Clone repository and install dependencies

In [ ]:
import os, sys, subprocess

GIT_URL = "https://github.com/SadreevAmir/sae-muc.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/sae-muc"

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("-> Installing dependencies ...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "numpy>=2.0.0,<2.1",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", "--no-cache-dir",
    "transformers>=4.40", "accelerate",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "sae-lens>=6.0", "pandas", "tqdm", "jsonlines", "huggingface_hub",
])

import torch
assert torch.cuda.is_available(), "GPU not available — change Runtime type to GPU!"
print(f"\nREPO: {REPO_DIR}")
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. (Optional) Google Drive

In [ ]:
MOUNT_DRIVE = False  # @param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
else:
    print("Drive not mounted — will use file upload in section 3.")

## 3. Upload files

1. **test.csv** — CSV with a `question` column
2. **intervention_v2.pt** — intervention config

In [ ]:
import os

UPLOAD_FILES = True  # @param {type:"boolean"}

UPLOAD_DIR = "/content/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

_uploaded_csv = None
_uploaded_pt = None

if UPLOAD_FILES:
    from google.colab import files

    print("=" * 60)
    print("Step 1/2: Upload test.csv")
    print("  Required column: question")
    print("=" * 60)
    up1 = files.upload()
    for fn, data in up1.items():
        dst = os.path.join(UPLOAD_DIR, fn)
        with open(dst, "wb") as f:
            f.write(data)
        if fn.endswith(".csv"):
            _uploaded_csv = dst
            print(f"  -> Saved CSV: {dst}")

    print()
    print("=" * 60)
    print("Step 2/2: Upload intervention_v2.pt")
    print("=" * 60)
    up2 = files.upload()
    for fn, data in up2.items():
        dst = os.path.join(UPLOAD_DIR, fn)
        with open(dst, "wb") as f:
            f.write(data)
        if fn.endswith(".pt"):
            _uploaded_pt = dst
            print(f"  -> Saved PT: {dst}")

    if _uploaded_csv:
        print(f"\nCSV: {_uploaded_csv}")
    if _uploaded_pt:
        print(f"PT:  {_uploaded_pt}")
else:
    print("Upload skipped — set paths manually in section 4.")

## 4. Configuration

**Alpha = 1.0 — hardcoded.** All questions receive the same intervention strength.

In [ ]:
# ── File paths ──
TEST_CSV         = _uploaded_csv or "/content/uploads/test.csv"            # @param {type:"string"}
INTERVENTION_PT  = _uploaded_pt  or "/content/uploads/intervention_v2.pt"  # @param {type:"string"}

# ── Model ──
MODEL_NAME       = "Mistral-7B-Instruct-v0.3"  # @param {type:"string"}
SAE_DTYPE        = "float32"                    # @param ["float32", "float16", "bfloat16"]

# ── Alpha (FIXED) ──
ALPHA            = 1.0

# ── Generation ──
GEN_BATCH_SIZE   = 4      # @param {type:"integer"}
APPLY_DURING_GEN = True   # @param {type:"boolean"}

# ── Debug ──
N_DEBUG          = 5      # @param {type:"integer"}

# ── Output ──
OUTPUT_DIR       = "/content/sae_clamp_results"  # @param {type:"string"}

# ── Validate ──
import os, pandas as pd
assert os.path.isfile(TEST_CSV),        f"CSV not found: {TEST_CSV}"
assert os.path.isfile(INTERVENTION_PT), f"PT not found: {INTERVENTION_PT}"

df_preview = pd.read_csv(TEST_CSV)
assert "question" in df_preview.columns, "CSV must have a 'question' column"

print(f"CSV:             {TEST_CSV}  ({len(df_preview)} rows)")
print(f"Intervention:    {INTERVENTION_PT}")
print(f"Model:           {MODEL_NAME}")
print(f"Alpha:           {ALPHA} (fixed for all questions)")
print(f"Apply during gen:{APPLY_DURING_GEN}")
print(f"Debug:           {N_DEBUG} questions")
print(f"Output dir:      {OUTPUT_DIR}")
print()
df_preview.head(5)

## 5. Hugging Face Login

In [ ]:
from huggingface_hub import login
login()

## 6. Load model, SAE and clamp config

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sae_lens import SAE

import sys, os
REPO_DIR = "/content/sae-muc"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from sae_muc.hooks import register_sae_clamp_hooks, clear_sae_latent_hooks
from sae_muc.generation import generate_lines_for_batch
from sae_muc.prompts_mini import make_sentence_user_content

torch.manual_seed(42)
np.random.seed(42)

# ── Resolve model name ──
if "Mistral" in MODEL_NAME:
    full_model_name = f"mistralai/{MODEL_NAME}"
elif "Llama" in MODEL_NAME:
    full_model_name = f"meta-llama/{MODEL_NAME}"
elif "Qwen" in MODEL_NAME:
    full_model_name = f"Qwen/{MODEL_NAME}"
else:
    full_model_name = MODEL_NAME

# ── Load CSV ──
print("Loading CSV ...")
df = pd.read_csv(TEST_CSV)
questions = df["question"].astype(str).tolist()
print(f"Questions: {len(questions)}")

messages = [[{"role": "user", "content": make_sentence_user_content(q)}] for q in questions]

# ── Load intervention_v2.pt (only clamp data) ──
print("\nLoading intervention config ...")
intervention_data = torch.load(INTERVENTION_PT, map_location="cpu", weights_only=False)
release = intervention_data["release"]
raw_layers = intervention_data["layers"]

sample_layer = next(iter(raw_layers.values()))
assert "method_clamp" in sample_layer, "intervention_v2.pt does not contain clamp method!"

print(f"  Release: {release}")
print(f"  Layers: {sorted(int(k) for k in raw_layers.keys())}")

# ── Extract clamp config ──
layer_to_clamp = {}
for k, v in raw_layers.items():
    hf_layer = int(k)
    clamp = v["method_clamp"]
    unc_idx = clamp["uncertainty_features"]
    cert_idx = clamp["certainty_features"]
    target_vals = clamp["target_uncertain_values"]
    layer_to_clamp[hf_layer] = {
        "unc_indices": torch.tensor(unc_idx, dtype=torch.long),
        "unc_targets": torch.tensor(
            [target_vals[i] for i in unc_idx], dtype=torch.float32
        ),
        "cert_indices": torch.tensor(cert_idx, dtype=torch.long),
    }

for l, cc in layer_to_clamp.items():
    print(f"  Clamp L{l}: {cc['unc_indices'].numel()} unc features, {cc['cert_indices'].numel()} cert features")

# ── Load SAEs ──
print("\nLoading SAEs ...")
layer_to_sae = {}
for k, v in raw_layers.items():
    hf_layer = int(k)
    sae_id = v["sae_id"]
    t0 = time.time()
    sae = SAE.from_pretrained(release=release, sae_id=sae_id, device="cpu", dtype=SAE_DTYPE)
    layer_to_sae[hf_layer] = sae
    print(f"  Layer {hf_layer} ({sae_id}): d_sae={sae.cfg.d_sae}, loaded in {time.time()-t0:.1f}s")

hook_layers = sorted(layer_to_sae.keys())

# ── Load LLM ──
print(f"\nLoading {full_model_name} (fp16, device_map=auto) ...")
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    full_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(full_model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"\nReady: {len(questions)} questions, {len(hook_layers)} SAE layers, alpha={ALPHA}")

## 7. Debug: first N questions (baseline vs clamp)

Quick run for visual inspection.

In [ ]:
from tqdm.auto import tqdm

n_dbg = min(N_DEBUG, len(questions))
dbg_questions = questions[:n_dbg]
dbg_messages = messages[:n_dbg]

print(f"Debug: {n_dbg} questions, alpha={ALPHA} (fixed)")
print("=" * 80)

# ── Baseline ──
print("\nGenerating BASELINE ...")
clear_sae_latent_hooks(model)
baseline_lines = generate_lines_for_batch(
    model, tokenizer, dbg_questions, dbg_messages, 0.0,
    greedy_only=True,
)

# ── Clamp (alpha=1) ──
print(f"Generating CLAMP (alpha={ALPHA}) ...")
clear_sae_latent_hooks(model)
register_sae_clamp_hooks(
    model, layer_to_sae, layer_to_clamp, hook_layers, ALPHA,
    apply_during_generation=APPLY_DURING_GEN,
)
clamp_lines = generate_lines_for_batch(
    model, tokenizer, dbg_questions, dbg_messages, ALPHA,
    greedy_only=True,
)
clear_sae_latent_hooks(model)

# ── Display ──
print("\n" + "=" * 80)
print("DEBUG RESULTS")
print("=" * 80)

for i in range(n_dbg):
    q = dbg_questions[i]
    print(f"\n{'─' * 80}")
    print(f"Q{i+1}: {q}")
    print(f"{'─' * 80}")
    base_ans = baseline_lines[i]["most_likely_answer"][:200]
    clamp_ans = clamp_lines[i]["most_likely_answer"][:200]
    print(f"  {'BASELINE':<16s} {base_ans}")
    print(f"  {'CLAMP a=1':<16s} {clamp_ans}")

print(f"\n{'=' * 80}")
print("Debug done.")

## 8. Full generation (clamp, alpha=1)

All questions, single pass, `alpha = 1`.

In [ ]:
from tqdm.auto import tqdm
import json, time

print(f"Full generation: {len(questions)} questions, method=clamp, alpha={ALPHA}")
print(f"Apply during generation: {APPLY_DURING_GEN}")
print(f"Mode: greedy only (most_likely_answer)")
print()

batch_size = max(1, GEN_BATCH_SIZE)
t0 = time.time()

# Register hooks once (same alpha for all questions)
clear_sae_latent_hooks(model)
register_sae_clamp_hooks(
    model, layer_to_sae, layer_to_clamp, hook_layers, ALPHA,
    apply_during_generation=APPLY_DURING_GEN,
)

results = []
for j in tqdm(range(0, len(questions), batch_size), desc=f"clamp a={ALPHA}"):
    batch_q = questions[j : j + batch_size]
    batch_m = messages[j : j + batch_size]
    lines = generate_lines_for_batch(
        model, tokenizer, batch_q, batch_m, ALPHA, greedy_only=True,
    )
    results.extend(lines)

clear_sae_latent_hooks(model)
torch.cuda.empty_cache()

print(f"\nDone: {len(results)} rows in {time.time()-t0:.0f}s")

## 9. Save and download

In [ ]:
import json
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"sae_clamp_alpha{ALPHA}.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved: {out_path}  ({len(results)} lines)")

# Preview
print(f"\n{'=' * 60}")
print(f"Preview (first 3 rows)")
print(f"{'=' * 60}")
for i, item in enumerate(results[:3]):
    print(f"  [{i}] alpha={item['alpha']}")
    print(f"      Q: {item['question'][:80]}")
    ans = item['most_likely_answer'][:120] if item['most_likely_answer'] else '(empty)'
    print(f"      A: {ans}")

# Stats
import pandas as pd
df_out = pd.DataFrame(results)
df_out["answer_len"] = df_out["most_likely_answer"].str.len()
print(f"\nOverall: {len(df_out)} rows, "
      f"answer_len: mean={df_out['answer_len'].mean():.0f}, "
      f"min={df_out['answer_len'].min()}, max={df_out['answer_len'].max()}")

# Download
try:
    from google.colab import files
    files.download(str(out_path))
    print(f"Download started: {out_path.name}")
except ImportError:
    print(f"\nNot in Colab — file saved in: {out_dir}")